# 2.7 — Jobs, Stages and Tasks

**Chapter 2, section 2.8** (*Narrow and Wide Dependencies*, *Jobs, Stages, and Tasks*).

**The question this notebook answers:** how many times does this pipeline read the data, and
where exactly does the shuffle fall?

The chapter's vocabulary is short and precise:

* **Job** — one action produces one job.
* **Stage** — a job is cut into stages **at every wide dependency**. All the narrow
  transformations between two shuffles are fused into a single stage and run together, record
  by record, without the intermediate RDDs ever being materialized.
* **Task** — one per partition, within a stage. Tasks are what is shipped to executors.

So **stage boundaries are shuffles**, and counting them is how you find out what a pipeline
costs. This notebook makes all three visible on Exercise 2's own pipeline, using
`toDebugString()` and Spark's own job metrics.

Covers **Exercise 2** in full, including the second half — *add a single line that would make
the second action much cheaper, and explain why it helps* — whose honest answer turns out to
be more interesting than the expected one.

Runs on a laptop in well under a minute.

In [1]:
# --- CS-777 session setup ------------------------------------------------
# Chapter 2, section 2.8.
import os, json, time, tempfile, urllib.request
from pyspark.sql import SparkSession

DATA = os.environ.get("CS777_DATA", "../data")          # -> code/data/
SCRATCH = os.environ.get("CS777_SCRATCH", os.path.join(tempfile.gettempdir(), "cs777"))
os.makedirs(SCRATCH, exist_ok=True)

spark = (SparkSession.builder
         .appName("CS777-2.7")
         .master("local[*]")
         .config("spark.sql.warehouse.dir", os.path.join(SCRATCH, "warehouse"))
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

sc = spark.sparkContext
UI = sc.uiWebUrl
APP = json.load(urllib.request.urlopen(UI + "/api/v1/applications"))[0]["id"]

def _get(path):
    return json.load(urllib.request.urlopen(f"{UI}/api/v1/applications/{APP}{path}"))

def run(label, fn):
    """Run an action under a named job group and report what Spark actually did.

    Everything here is read from this session's own Spark UI over its REST API -- the
    same numbers the Jobs and Stages tabs display, captured so they survive into the
    saved notebook.  Nothing outside this machine is contacted.
    """
    sc.setJobGroup(label, label)
    t = time.perf_counter()
    out = fn()
    ms = (time.perf_counter() - t) * 1000
    jobs = [j for j in _get("/jobs") if j.get("jobGroup") == label]
    return {"label": label, "ms": ms, "result": out,
            "jobs": len(jobs),
            "stages": sum(len(j["stageIds"]) for j in jobs),
            "skipped_stages": sum(j["numSkippedStages"] for j in jobs),
            "tasks_run": sum(j["numCompletedTasks"] for j in jobs),
            "tasks_skipped": sum(j["numSkippedTasks"] for j in jobs)}

def show(*rs):
    print(f"{'action':14s}{'ms':>8s}{'jobs':>6s}{'stages':>8s}{'skipped':>9s}"
          f"{'tasks run':>11s}{'tasks skipped':>15s}")
    for r in rs:
        print(f"{r['label']:14s}{r['ms']:>8.0f}{r['jobs']:>6}{r['stages']:>8}"
              f"{r['skipped_stages']:>9}{r['tasks_run']:>11}{r['tasks_skipped']:>15}")

print("Spark", spark.version, "on", sc.master)

Spark 4.2.0 on local[*]


## 1. Exercise 2's pipeline

> ```python
> rdd    = sc.textFile("gs://bucket/data.txt")
> words  = rdd.flatMap(lambda line: line.split())
> pairs  = words.map(lambda w: (w, 1))
> counts = pairs.reduceByKey(lambda a, b: a + b)
> top    = counts.filter(lambda kv: kv[1] > 100)
> print(top.count())
> print(top.take(5))
> ```
>
> *Say how many jobs run, how many stages each job has, and where the stage boundaries fall.*

The only change below is the input: a local corpus rather than an object-store URI, so that
the notebook needs no network. The threshold is lowered to 20 because the corpus is smaller.

In [2]:
rdd = sc.textFile(os.path.join(DATA, "Alices-Adventures-in-Wonderland-by-Lewis-Carroll.txt.bz2"))
words = rdd.flatMap(lambda line: line.split())
pairs = words.map(lambda w: (w, 1))
counts = pairs.reduceByKey(lambda a, b: a + b)
top = counts.filter(lambda kv: kv[1] > 20)

print("partitions in the input:", rdd.getNumPartitions())
print("\nNothing has run yet.  Five transformations, no action, no job.")
print("jobs so far:", len(_get("/jobs")))

partitions in the input: 2

Nothing has run yet.  Five transformations, no action, no job.
jobs so far: 0


## 2. Where the boundary is, before running anything

`toDebugString()` prints the lineage. Read it from the bottom up: it is the recipe, and the
indentation marks the stage boundaries.

In [3]:
print(top.toDebugString().decode())

(2) PythonRDD[6] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[5] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[4] at partitionBy at DirectMethodHandleAccessor.java:103 []
 +-(2) PairwiseRDD[3] at reduceByKey at /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/ipykernel_34615/1817031215.py:4 []
    |  PythonRDD[2] at reduceByKey at /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/ipykernel_34615/1817031215.py:4 []
    |  ../data/Alices-Adventures-in-Wonderland-by-Lewis-Carroll.txt.bz2 MapPartitionsRDD[1] at textFile at DirectMethodHandleAccessor.java:103 []
    |  ../data/Alices-Adventures-in-Wonderland-by-Lewis-Carroll.txt.bz2 HadoopRDD[0] at textFile at DirectMethodHandleAccessor.java:103 []


Two things to read out of that.

The number in parentheses at the start of a line — `(2)` — is the **partition count** of that
RDD, and therefore the number of tasks in its stage.

The `+-` and the indent step mark a **stage boundary**. Everything below it, from `HadoopRDD`
through `textFile`, `flatMap` and `map`, is one stage: those are narrow, so Spark fuses them
and runs them record by record without ever building the intermediate RDDs. The `ShuffledRDD`
above the boundary is the shuffle that `reduceByKey` requires, and the `filter` after it is
narrow again and gets fused into the second stage.

So: **one shuffle, two stages**. That is the answer to the first half of Exercise 2, and it did
not require running anything.

## 3. What each action actually does

In [4]:
a = run("count", lambda: top.count())
print("count() =", a["result"])
show(a)
print("\nOne action, one job, two stages -- as the lineage predicted.")
print(f"{a['tasks_run']} tasks ran: {rdd.getNumPartitions()} in the read stage and the rest")
print("in the stage after the shuffle.")

count() = 193
action              ms  jobs  stages  skipped  tasks run  tasks skipped
count              620     1       2        0          4              0

One action, one job, two stages -- as the lineage predicted.
4 tasks ran: 2 in the read stage and the rest
in the stage after the shuffle.


In [5]:
b = run("take", lambda: top.take(5))
print("take(5) =", b["result"])
show(a, b)

take(5) = [('Gutenberg', 21), ('of', 604), ('by', 81), ('for', 146), ('use', 24)]
action              ms  jobs  stages  skipped  tasks run  tasks skipped
count              620     1       2        0          4              0
take                17     1       2        1          1              2


### The surprise, and the honest answer to the second half of Exercise 2

The second action was a great deal cheaper than the first, and **nothing was cached**. Look at
the `skipped` column: Spark skipped an entire stage.

The reason is that a shuffle writes its output to the executors' local disks, and those files
outlive the job that produced them. When a second job needs the same shuffle, Spark notices
the output is still there and skips everything upstream of it. This is automatic, it is not
caching, and it is why the Spark UI shows stages marked *skipped* in almost every real
application.

That complicates the exercise's expected answer, `counts.cache()`. Caching *is* the right
answer, but not for the reason the phrasing suggests, and the improvement it buys here is
small — Spark had already removed most of the duplicated work. The place where `cache()` is
decisive is where there is **no shuffle to reuse**, and section 5 shows that case.

## 4. Tasks, and why the count is what it is

One task per partition, per stage. The partition count of the input therefore sets the width
of the first stage, and `spark.sql.shuffle.partitions` — or the argument to `reduceByKey` —
sets the width of the second.

In [6]:
for n in (2, 8):
    tight = (rdd.flatMap(lambda line: line.split())
                .map(lambda w: (w, 1))
                .reduceByKey(lambda x, y: x + y, numPartitions=n)
                .filter(lambda kv: kv[1] > 20))
    r = run(f"reduce-{n}", lambda: tight.count())
    print(f"reduceByKey(numPartitions={n:>2}) -> {r['tasks_run']} tasks over "
          f"{r['stages']} stages, {r['ms']:.0f} ms")

print("\nThe first stage is fixed by the input; only the second changes.  A stage over 200")
print("partitions is 200 tasks, and that is the only sense in which Spark's parallelism")
print("is a number you choose.")

reduceByKey(numPartitions= 2) -> 4 tasks over 2 stages, 58 ms


reduceByKey(numPartitions= 8) -> 10 tasks over 2 stages, 159 ms

The first stage is fixed by the input; only the second changes.  A stage over 200
partitions is 200 tasks, and that is the only sense in which Spark's parallelism
is a number you choose.


## 5. Where caching actually earns its place

Two actions over a pipeline with **no shuffle in it**. There is no shuffle output to reuse, so
Spark honestly does the whole thing twice: it decompresses and re-reads the file both times.

In [7]:
plain = (sc.textFile(os.path.join(DATA,
                     "Alices-Adventures-in-Wonderland-by-Lewis-Carroll.txt.bz2"))
           .flatMap(lambda line: line.split())
           .filter(lambda w: len(w) > 4))

c1 = run("nocache-1", lambda: plain.count())
c2 = run("nocache-2", lambda: plain.count())
show(c1, c2)
print("\nSame work, twice.  No stage was skipped, because there was no shuffle output to")
print("skip past.  This is what 'two actions run the pipeline twice' actually means.")

action              ms  jobs  stages  skipped  tasks run  tasks skipped
nocache-1           24     1       1        0          2              0
nocache-2           25     1       1        0          2              0

Same work, twice.  No stage was skipped, because there was no shuffle output to
skip past.  This is what 'two actions run the pipeline twice' actually means.


In [8]:
plain.cache()
_ = run("warm", lambda: plain.count())        # the run that fills the cache

d1 = run("cached-1", lambda: plain.count())
d2 = run("cached-2", lambda: plain.count())
show(c2, d1, d2)
print(f"\nspeed-up after caching: {c2['ms'] / max(d2['ms'], 0.001):.1f}x")
print("\nThat is the line Exercise 2 is asking for, and this is the shape of pipeline")
print("where it matters: one that is re-read from source rather than re-read from a")
print("shuffle.  cache() is not free -- it occupies executor memory, and Spark will")
print("evict it under pressure -- so it is worth adding where a measurement says so.")

action              ms  jobs  stages  skipped  tasks run  tasks skipped
nocache-2           25     1       1        0          2              0
cached-1            15     1       1        0          2              0
cached-2            13     1       1        0          2              0

speed-up after caching: 2.0x

That is the line Exercise 2 is asking for, and this is the shape of pipeline
where it matters: one that is re-read from source rather than re-read from a
shuffle.  cache() is not free -- it occupies executor memory, and Spark will
evict it under pressure -- so it is worth adding where a measurement says so.


In [9]:
# What Spark now holds.  The default storage level for an RDD is MEMORY_ONLY: if it does
# not fit, the parts that do not are simply recomputed on demand rather than spilled.
for r in _get("/storage/rdd"):
    print(f"RDD {r['id']:>3}  {r['name'][:40]:40s}  {r['storageLevel']}")
    print(f"        cached partitions {r['numCachedPartitions']}/{r['numPartitions']}"
          f"   in memory {r['memoryUsed'] / 1024:,.0f} KB")

plain.unpersist()
print("\nunpersisted")

RDD  23  PythonRDD                                 Memory Serialized 1x Replicated
        cached partitions 2/2   in memory 66 KB

unpersisted


## 6. When the count is not the whole story

The chapter is careful about this:

> In a job whose stages form a single chain, five stages in the Spark UI indicate four
> shuffles; where the job branches, as a join makes it, read the graph rather than the count.

A join is the usual reason a stage graph stops being a line: both sides have to be shuffled so
that matching keys meet, so the stage that performs the join has **two parents**. Below is the
same join at both levels of the API, and they do not behave the same way — which is exactly why
the chapter says to read the graph.

In [10]:
left = sc.parallelize([(i % 50, f"L{i}") for i in range(2000)], 4)
right = sc.parallelize([(i % 50, f"R{i}") for i in range(2000)], 4)
joined = left.join(right).filter(lambda kv: kv[0] % 7 == 0)

print(joined.toDebugString().decode())

(8) PythonRDD[36] at RDD at PythonRDD.scala:59 []
 |  MapPartitionsRDD[35] at mapPartitions at PythonRDD.scala:197 []
 |  ShuffledRDD[34] at partitionBy at DirectMethodHandleAccessor.java:103 []
 +-(8) PairwiseRDD[33] at join at /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/ipykernel_34615/4116731164.py:3 []
    |  PythonRDD[32] at join at /var/folders/wh/ptq7zytj1gz42sqc0rs52tm40000gn/T/ipykernel_34615/4116731164.py:3 []
    |  UnionRDD[31] at union at DirectMethodHandleAccessor.java:103 []
    |  PythonRDD[29] at RDD at PythonRDD.scala:59 []
    |  ParallelCollectionRDD[27] at readRDDFromFile at PythonRDD.scala:326 []
    |  PythonRDD[30] at RDD at PythonRDD.scala:59 []
    |  ParallelCollectionRDD[28] at readRDDFromFile at PythonRDD.scala:326 []


In [11]:
j = run("rdd-join", lambda: joined.count())
show(j)

action              ms  jobs  stages  skipped  tasks run  tasks skipped
rdd-join           156     1       2        0         16              0


### The RDD join is a chain, not a Y

Two stages, not three or four — and the lineage above says why. **PySpark implements an RDD
join as a `union` of the two sides followed by a single `partitionBy`.** Look for `UnionRDD` in
the debug string: both parents feed into it, and a union is a *narrow* dependency, so both
sides are read inside the **same** stage, as eight tasks over the two RDDs' four partitions
each. Then one shuffle brings the keys together.

So at the RDD level the chain arithmetic does hold: two stages, one shuffle. That is worth
knowing before you go looking for a branch that is not there.

In [12]:
# The DataFrame join is the branching case.  Broadcasting and adaptive execution are
# turned off so that the plan is the plain sort-merge join the chapter describes,
# rather than the optimizer's preferred shortcut for tables this small.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.enabled", "false")

L = spark.createDataFrame([(i % 50, f"L{i}") for i in range(2000)], "k int, l string").repartition(4)
R = spark.createDataFrame([(i % 50, f"R{i}") for i in range(2000)], "k int, r string").repartition(4)

d = run("df-join", lambda: L.join(R, "k").filter("k % 7 = 0").count())
print("rows:", d["result"])
show(j, d)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")

rows: 12800
action              ms  jobs  stages  skipped  tasks run  tasks skipped
rdd-join           156     1       2        0         16              0
df-join           1005     1       6        0        245              0


**One action, one job, six stages** — and nowhere near five shuffles. Each side of the join is
built and shuffled in its own pair of stages, and a further stage joins them and counts. The
graph is a Y with a stem, and reading "six stages therefore five shuffles" off it would be
wrong in both directions.

This is the case the chapter is warning about, and it is the one a reader will actually meet,
because DataFrames are where joins are written from chapter 3 onwards. The Spark UI draws this
graph on the job's own page; the stage *count* is not a substitute for looking at it.

## Conclusion

**One action, one job.** Two actions run the pipeline twice — unless Spark can reuse something,
which is the whole subject of sections 3 and 5.

**Stage boundaries are shuffles.** `toDebugString()` shows them before anything runs, as an
indent step; the Spark UI shows them after. Narrow transformations between two shuffles are
fused into one stage and pipelined, which is a direct dividend of laziness: Spark could only
fuse them because it saw all of them before running any.

**One task per partition.** A stage over 200 partitions is 200 tasks, and the number of tasks
that can run at once is the number of cores, not the number of partitions.

**A shuffle is a barrier.** No downstream task can start until every upstream task has
finished, because any upstream task might still contribute records to any downstream partition.
That is why one slow task delays not just its own stage but everything after it — the straggler
of chapter 1, and the skew of notebook 2.4.

Two habits worth taking from this notebook:

1. **Read `toDebugString()` before running anything expensive.** The number of stage boundaries
   in it is the number of shuffles you are about to pay for, and it costs nothing to look.
2. **Check the `skipped` column before reaching for `cache()`.** Spark reuses shuffle output
   automatically. Cache buys you the *un*-shuffled part of a lineage, and if there is not one,
   it buys very little.

**Next.** Notebook 2.8 removes a shuffle entirely, which is the highest-value optimization in
the chapter.